In [ ]:
import pandas as pd, numpy as np
from wgans import *
import sys; sys.path.insert(0, '../')
from util import *

SEEDS  = [1, 2, 3, 4, 5]
SCENES = [1, 2, 3, 4, 5, 6]              # scene0 (full) .. scene6 (10/class)
RATIOS = [0.0, 0.5, 1.0, 2.0]   # 0.0 = no-augmentation baseline

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)


# Helper function

In [ ]:
def preprocess_scenario(train, verbose=False):
    # Step 1: outlier removal on training only
    # train_outlier_removed, outlier_log = outlier_remove(train, verbose=verbose)
    
    # Step 2: Z-score — fit on train, apply to all
    X_train_norm, scaler = zscore(train)
    # X_val_norm,   _      = zscore(val,  scaler=scaler)
    # X_test_norm,  _      = zscore(test, scaler=scaler)
    
    return X_train_norm, scaler



# Train and generate GANs

In [ ]:
for seed in SEEDS:
    for scene in SCENES:
        print(f"seed{seed} scene{scene}")
        # Load the scenario
        train_arr = pd.read_csv(f"../CSV_Files/seed{seed}/scene{scene}/train.csv").to_numpy()
        X_tr_n, _ = preprocess_scenario(train_arr)

        # Train the DCGAN on the training data
        dcgans, dcgans_hist = train_class_gans(train_arr, seed=seed, device=device, n_epochs = 1000, lr=1e-3,verbose=True)

        # Generate synthetic data using the trained DCGAN
        for ratio in RATIOS:
            aug = augment(train_arr, dcgans, ratio, seed=seed, device=device)

            # Save the augmented training data to a new CSV file
            augmented_df = pd.DataFrame(aug, columns=[f"feature_{i}" for i in range(1, 16)])
            augmented_df.to_csv(f"../CSV_Files/seed{seed}/scene{scene}/train_wgan_augmented_ratio{ratio}.csv", index=False)
